# Project Alpha: RL Bot v63p

## 📋 TODO & Roadmap
- **Data Policy:** Should we apply `handle_zeros_as_nan` globally to all matrices?
- **Feature Scaling:** Finalize the `get_agent_view` dual-view scaling logic.
- **Performance:** Move heavy logic (`analyze_4d_regime`) into `core` modular library.

<details>
<summary><b>📜 Version History (v59 - v63)</b></summary>

### v63
- Added **Metric Registry** (Strong-typed, dual-view).
- Fixed **Index Poisoning** bug (Dates in Ticker column).
- Unified **Perception** and **Execution** in Audit packs.
- Added 4 New Pillars: Return Autocorr, Range Position, OBV Divergence, Convexity.

### v62
- Fixed **Object Reference Mutation** bug in `compute_alpha_ensemble`.
- Added `PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`.

### v61
- Verified all metrics with Excel parity.
- Refactored `AlphaEngine` into Orchestrator pattern.

### v60
- Converted notebook code to modular system.
- Added Volatility Regime plots.

### v59
- Implemented **Result Pattern** for error handling.
- Switched to logarithmic returns globally.
</details>

## I. Initialization & Environment

In [1]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pandas as pd
import numpy as np
import multiprocessing
import random
import re
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
from types import SimpleNamespace
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import plotly.io as pio

def add_project_root_to_path():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            return path
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not find notebooks_RLVR directory")

add_project_root_to_path()

# 2. Display Settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.precision", 4)

# 3. Project Imports
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput, MarketObservation
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.quant import QuantUtils, TickerEngine
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import STRATEGY_REGISTRY

# 4. Load Environment Paths
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"✓ Paths Initialized.\n  OHLCV: {DATA_PATH_OHLCV}\n  Indices: {DATA_PATH_INDICES}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

✓ Paths Initialized.
  OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
  Indices: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


## II. Data Pipeline

In [2]:
# === RELOADING FROM PARQUET ===
features_df = pd.read_parquet("output/features_df.parquet")
macro_df = pd.read_parquet("output/macro_df.parquet")
df_close_wide = pd.read_parquet("output/df_close_wide.parquet")
df_atrp_wide = pd.read_parquet("output/df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet("output/df_trp_wide.parquet")

df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

In [ ]:
print("⏳ Loading DataFrames... takes ~6 minutes")
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    OUTPUT_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
)

print("⚡ Generating Features...")
features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    df_fed=df_fed,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

print("🚀 Unstacking Wide Matrices...")
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

In [ ]:
# === PERSISTENCE SANDBOX ===
features_df.to_parquet("output/features_df.parquet", index=True)
macro_df.to_parquet("output/macro_df.parquet", index=True)
df_close_wide.to_parquet("output/df_close_wide.parquet", index=True)
df_atrp_wide.to_parquet("output/df_atrp_wide.parquet", index=True)
df_trp_wide.to_parquet("output/df_trp_wide.parquet", index=True)

## III. The Analysis Suite

In [3]:
# SINGLE SOURCE OF TRUTH
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [ ]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=False,
)

analyzer1, _ = create_walk_forward_analyzer(engine, _inputs, universe_subset=None)
analyzer1.show()

DEBUG: 941 stocks passed filters on 2026-04-16


In [ ]:
##################################
##################################
##################################

In [7]:
df_ohlcv.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9515972 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-04-24 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 400.0+ MB


In [52]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 20))
[  4]   📂 tickers (len=20)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]     📄 index_10 (str)
[ 16]     📄 index_11 (str)
[ 17]     📄 index_12 (str)
[ 18]     📄 index_13 (str)
[ 19]     📄 index_14 (str)
[ 20]     📄 index_15 (str)
[ 21]     📄 index_16 (str)
[ 22]     📄 index_17 (str)
[ 23]     📄 index_18 (str)
[ 24]     📄 index_19 (str)
[ 25]   📈 initial_weights (shape=(20,))
[ 26]   📂 perf_metrics (len=24)
[ 27]     🔢 full_p_gain (float)
[ 28]     🔢 full_p_sharpe (float)
[ 29]     🔢 full_p_sharpe_atrp (float)
[ 30]     🔢 full_p_sharpe_trp (float)
[ 31]     🔢 lookback_p_gain (float

In [ ]:
# # Set the first index here
# base_idx = 35

# # Unpack the next 4 consecutive values
# start_date, decision_date, buy_date, holding_end_date = [
#     pd.Timestamp(SU.peek(i, result)) for i in range(base_idx, base_idx + 4)
# ]

# Print to verify
print(f"Start:    {start_date.date()}")
print(f"Decision: {decision_date.date()}")
print(f"Buy:      {buy_date.date()}")
print(f"End:      {holding_end_date.date()}")

Start:    2026-04-01
Decision: 2026-04-16
Buy:      2026-04-17
End:      2026-04-24


In [58]:
target_tickers = SU.peek(4, result)

 📍 INDEX: [4]
 🏷️  NAME:  tickers
 📂 PATH:  audit_pack -> tickers



['SOXL',
 'CRDO',
 'NBIS',
 'IONQ',
 'ALAB',
 'BE',
 'QBTS',
 'CRWV',
 'RVMD',
 'RGTI',
 'INTC',
 'TECL',
 'IREN',
 'CIFR',
 'HIMS',
 'OKLO',
 'RMBS',
 'SNDK',
 'CLS',
 'AMD']

In [ ]:
# start_date = "2020-01-01"
end_date = decision_date

# Create the index slicer
idx = pd.IndexSlice

# Slice: (All Tickers, Date Range)
df_portf_close = df_ohlcv.loc[idx[target_tickers, start_date:end_date], ["Adj Close"]]

# 1. Calculate the first available price for each ticker in this slice
# transform('first') broadcasts the first price to every row for that ticker
first_prices = df_portf_close.groupby(level=0)["Adj Close"].transform("first")

# 2. Divide by the first prices to normalize
# We create a new DataFrame or overwrite the column
df_portf_norm = (df_portf_close["Adj Close"] / first_prices).to_frame(
    name="Normalized Close"
)

print(f"df_portf_close:\n{df_portf_close}\n")
print(f"df_portf_norm:\n{df_portf_norm}\n")

df_portf_close:
                   Adj Close
Ticker Date                 
SOXL   2026-04-01      52.26
       2026-04-02      52.75
       2026-04-06      54.81
       2026-04-07      56.55
       2026-04-08      67.50
...                      ...
AMD    2026-04-10     245.04
       2026-04-13     246.83
       2026-04-14     255.07
       2026-04-15     258.12
       2026-04-16     278.26

[220 rows x 1 columns]

df_portf_norm:
                   Normalized Close
Ticker Date                        
SOXL   2026-04-01            1.0000
       2026-04-02            1.0094
       2026-04-06            1.0488
       2026-04-07            1.0821
       2026-04-08            1.2916
...                             ...
AMD    2026-04-10            1.1657
       2026-04-13            1.1742
       2026-04-14            1.2134
       2026-04-15            1.2279
       2026-04-16            1.3237

[220 rows x 1 columns]



In [ ]:
portf_final_norm_price = df_portf_norm.groupby(level=0).last()
portfolio_log_gain = np.log(portf_final_norm_price.mean())

print(f"portf_final_norm_price:\n{portf_final_norm_price}\n")

# Use .iloc[0] to get the first (and only) value
print(f"portfolio_log_gain: {portfolio_log_gain.iloc[0]:.4f}")

portf_final_norm_price:
        Normalized Close
Ticker                  
ALAB              1.6064
AMD               1.3237
BE                1.5860
CIFR              1.3718
CLS               1.3241
CRDO              1.6569
CRWV              1.5242
HIMS              1.3604
INTC              1.4262
IONQ              1.6078
IREN              1.3992
NBIS              1.6218
OKLO              1.3358
QBTS              1.5708
RGTI              1.4407
RMBS              1.3344
RVMD              1.5133
SNDK              1.3273
SOXL              1.6910
TECL              1.4162

portfolio_log_gain: 0.3866


In [79]:
type(portfolio_log_gain)

pandas.core.series.Series

In [ ]:
# 2. Get the FINAL normalized price (Price Relative) for each ticker
# (Normalized starts at 1.0 for every ticker)
final_price_relatives = df_portf_norm.groupby(level=0)["Normalized"].last()

# 3. Calculate the Portfolio Return
# Average the relatives (this is the value of your portfolio if you started with $1 in each)
portfolio_value_end = final_price_relatives.mean()

# 4. Take the log of the final value
portfolio_log_gain = np.log(portfolio_value_end)

In [11]:
# start_date = "2020-01-01"
end_date = decision_date

# Create the index slicer
idx = pd.IndexSlice

# Slice: (All Tickers, Date Range)
df_decision = df_ohlcv.loc[idx[:, start_date:end_date], :]

In [61]:
# 1. Slice: (All Tickers, Date Range, ONLY Adj Close column)
# We use [['Adj Close']] to keep it as a DataFrame for easy column adding
idx = pd.IndexSlice
df_result = df_ohlcv.loc[idx[:, start_date:decision_date], ["Adj Close"]].copy()

# 2. Get the first price for each ticker (broadcasted to every row)
# This finds the price on the 'start_date' for every ticker
first_prices = df_result.groupby(level=0)["Adj Close"].transform("first")

# 3. Calculate Normalized Close (Price Relative: starts at 1.0)
# Formula: Price_t / Price_start
df_result["Normalized"] = df_result["Adj Close"] / first_prices

In [ ]:
# 1. Filter for your target tickers
df_targets = df_result.loc[pd.IndexSlice[target_tickers, :]]

# 2. Get the FINAL normalized price (Price Relative) for each ticker
# (Normalized starts at 1.0 for every ticker)
final_price_relatives = df_targets.groupby(level=0)["Normalized"].last()

# 3. Calculate the Portfolio Return
# Average the relatives (this is the value of your portfolio if you started with $1 in each)
portfolio_value_end = final_price_relatives.mean()

# 4. Take the log of the final value
portfolio_log_gain = np.log(portfolio_value_end)

print(f"df_targets:\n{df_targets}\n")
print(f"final_price_relatives:\n{final_price_relatives}\n")
print(f"portfolio_value_end:\n{portfolio_value_end}\n")
print(f"Buy & Hold Portfolio Log Gain: {portfolio_log_gain:.4f}")

df_targets:
                   Adj Close  Normalized
Ticker Date                             
SOXL   2026-04-01      52.26      1.0000
       2026-04-02      52.75      1.0094
       2026-04-06      54.81      1.0488
       2026-04-07      56.55      1.0821
       2026-04-08      67.50      1.2916
...                      ...         ...
AMD    2026-04-10     245.04      1.1657
       2026-04-13     246.83      1.1742
       2026-04-14     255.07      1.2134
       2026-04-15     258.12      1.2279
       2026-04-16     278.26      1.3237

[220 rows x 2 columns]

final_price_relatives:
Ticker
ALAB    1.6064
AMD     1.3237
BE      1.5860
CIFR    1.3718
CLS     1.3241
CRDO    1.6569
CRWV    1.5242
HIMS    1.3604
INTC    1.4262
IONQ    1.6078
IREN    1.3992
NBIS    1.6218
OKLO    1.3358
QBTS    1.5708
RGTI    1.4407
RMBS    1.3344
RVMD    1.5133
SNDK    1.3273
SOXL    1.6910
TECL    1.4162
Name: Normalized, dtype: float64

portfolio_value_end:
1.471901908334943

Buy & Hold Portfolio Log G

In [55]:
# 1. Slice: (All Tickers, Date Range, ONLY Adj Close column)
# We use [['Adj Close']] to keep it as a DataFrame for easy column adding
idx = pd.IndexSlice
df_result = df_ohlcv.loc[idx[:, start_date:decision_date], ["Adj Close"]].copy()

# 2. Get the first price for each ticker (broadcasted to every row)
# This finds the price on the 'start_date' for every ticker
first_prices = df_result.groupby(level=0)["Adj Close"].transform("first")

# 3. Calculate Normalized Close (Price Relative: starts at 1.0)
# Formula: Price_t / Price_start
df_result["Normalized"] = df_result["Adj Close"] / first_prices

# 4. Calculate Cumulative Log Return
# Formula: ln(Price_t / Price_start) which is the same as ln(Normalized)
with np.errstate(divide="ignore", invalid="ignore"):
    df_result["Log_Return"] = np.log(df_result["Normalized"])

# 5. Handle edge cases (Price was 0 or missing)
df_result["Log_Return"] = (
    df_result["Log_Return"].fillna(0.0).replace([np.inf, -np.inf], -10.0)
)

print(df_result.head())

                   Adj Close  Normalized  Log_Return
Ticker Date                                         
A      2026-04-01     114.54      1.0000      0.0000
       2026-04-02     115.48      1.0082      0.0082
       2026-04-06     114.84      1.0026      0.0026
       2026-04-07     113.88      0.9942     -0.0058
       2026-04-08     116.92      1.0208      0.0206


In [59]:
# # 1. Your list of specific tickers
# target_tickers = ["AAPL", "MSFT", "NVDA", "GOOGL"]  # Example list

# 2. Filter for these tickers and get the LAST Log_Return for each
# level=0 is the Ticker level
final_returns = (
    df_result.loc[pd.IndexSlice[target_tickers, :], "Log_Return"]
    .groupby(level=0)
    .last()
)

# 3. Calculate the mean
mean_log_return = final_returns.mean()

print(f"Mean Cumulative Log Return: {mean_log_return:.4f}")
print(f"Percentage Equivalent: {(np.exp(mean_log_return) - 1):.2%}")

Mean Cumulative Log Return: 0.3831
Percentage Equivalent: 46.68%


,Adj Close,Normalized,Log_Return
Date,,,
2026-04-01,114.54,1.0000,0.0000
2026-04-02,115.48,1.0082,0.0082
2026-04-06,114.84,1.0026,0.0026
2026-04-07,113.88,0.9942,-0.0058
2026-04-08,116.92,1.0208,0.0206
2026-04-09,115.39,1.0074,0.0074
2026-04-10,115.06,1.0045,0.0045
2026-04-13,117.50,1.0258,0.0255
2026-04-14,120.39,1.0511,0.0498


In [ ]:
df_decision.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 17358 entries, ('A', Timestamp('2026-04-01 00:00:00')) to ('ZWS', Timestamp('2026-04-16 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Open   17358 non-null  float64
 1   Adj High   17358 non-null  float64
 2   Adj Low    17358 non-null  float64
 3   Adj Close  17358 non-null  float64
 4   Volume     17358 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 1.4+ MB


In [21]:
print(df_decision.head())
print(df_decision.tail())

                   Adj Open  Adj High  Adj Low  Adj Close   Volume
Ticker Date                                                       
A      2026-04-01    114.01    115.72   113.85     114.54  1555300
       2026-04-02    113.91    117.32   112.99     115.48  1111900
       2026-04-06    115.14    115.68   113.02     114.84  1153700
       2026-04-07    113.94    114.55   112.90     113.88  1625000
       2026-04-08    115.98    118.25   115.98     116.92  1384600
                   Adj Open  Adj High  Adj Low  Adj Close   Volume
Ticker Date                                                       
ZWS    2026-04-10     48.23     48.23    47.66      47.74   609000
       2026-04-13     47.78     49.65    47.58      49.61  1526100
       2026-04-14     49.44     49.88    48.83      49.35  1019000
       2026-04-15     48.97     49.14    47.14      47.39   936400
       2026-04-16     47.35     48.07    46.35      46.51  1288500


In [ ]:
# Pass only 'Adj Close' to the function
ticker_gains = (
    df_decision["Adj Close"].groupby(level=0).apply(QuantUtils.calculate_gain)
)

# Convert Series to DataFrame and rename for clarity
df_final_gains = ticker_gains.to_frame(name="log_gain")

print(df_final_gains)

        log_gain
Ticker          
A         0.0319
AA       -0.0232
AAL       0.0975
AAON      0.0895
AAPL      0.0299
...          ...
ZM        0.0715
ZS       -0.0173
ZTO       0.0358
ZTS       0.0148
ZWS       0.0308

[1578 rows x 1 columns]


In [ ]:
def calculate_gain_vectorized(df, min_points=2):
    # 1. Group by Ticker (level 0) and get first/last valid values
    # This happens at C-speed, skipping NaNs automatically
    grouped = df.groupby(level=0)

    first_vals = grouped.first()
    last_vals = grouped.last()
    counts = grouped.count()

    # 2. Vectorized Log Return: ln(last / first)
    # This handles all tickers and all columns simultaneously
    with np.errstate(divide="ignore", invalid="ignore"):
        gain = np.log(last_vals / first_vals)

    # 3. Handle your business logic (min_points and non-positive prices)
    # Set to -10.0 if prices are <= 0 (matches your logic)
    gain[(first_vals <= 0) | (last_vals <= 0)] = -10.0

    # Set to 0.0 if not enough points
    gain[counts < min_points] = 0.0

    return gain.fillna(0.0)


#

In [43]:
log_gain_vec = calculate_gain_vectorized(df_decision["Adj Close"])
log_gain_vec

Ticker
A       0.0319
AA     -0.0232
AAL     0.0975
AAON    0.0895
AAPL    0.0299
         ...  
ZM      0.0715
ZS     -0.0173
ZTO     0.0358
ZTS     0.0148
ZWS     0.0308
Name: Adj Close, Length: 1578, dtype: float64

In [53]:
_tickers = SU.peek(4, result)

 📍 INDEX: [4]
 🏷️  NAME:  tickers
 📂 PATH:  audit_pack -> tickers



['SOXL',
 'CRDO',
 'NBIS',
 'IONQ',
 'ALAB',
 'BE',
 'QBTS',
 'CRWV',
 'RVMD',
 'RGTI',
 'INTC',
 'TECL',
 'IREN',
 'CIFR',
 'HIMS',
 'OKLO',
 'RMBS',
 'SNDK',
 'CLS',
 'AMD']

In [54]:
log_gain_vec.loc[_tickers].mean()

0.3830757648771967

In [ ]:
_ticker = "A"
_ser = df_decision.loc[_ticker]["Adj Close"]
log_gain = np.log(_ser.iloc[-1] / _ser.iloc[0])
print(f"{_ticker} log gain: {log_gain}")

A log gain: 0.03187692040633108


In [ ]:
##################################
##################################
##################################

In [ ]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
# Configuration for Interactive UI
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-04-16"),
    lookback_period=10,
    holding_period=5,
    metric="Log Price Gain",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=3,
    debug=True,
)

print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df)
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
DEBUG: 941 stocks passed filters on 2026-04-16
----------------------------------------------------------------------
Timeline: [2026-04-01] -> Decision: 2026-04-16 -> Entry: 2026-04-17 -> End: 2026-04-24
Selected Tickers (3):
SOXL, CRDO, NBIS
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.6818,0.5047,0.1557
Benchmark Gain,0.0858,0.0684,0.0053
== Gain Delta,0.5960,0.4363,0.1504
Group Sharpe,23.1077,24.1712,38.7616
Benchmark Sharpe,10.9495,14.0962,2.3401
== Sharpe Delta,12.1582,10.0750,36.4215
Group Sharpe (ATRP),0.6272,0.6934,0.5230
Benchmark Sharpe (ATRP),0.3892,0.4683,0.0879
== Sharpe (ATRP) Delta,0.2380,0.2251,0.4351



Target Reward Signal: 0.155737


## IV. Deep Dives & Verification

In [ ]:
def verify_alpha_integrity(engine: AlphaEngine, ticker: str, date: pd.Timestamp):
    """Manual audit of the 4 New Pillars (Autocorr, Range, OBV, Convexity)."""
    alpha_matrix = engine.compute_alpha_matrix(decision_date=date, lookback_period=21)
    if ticker not in alpha_matrix.index:
        return print("❌ Ticker filtered out.")

    # --- Autocorr ---
    engine_val = alpha_matrix.loc[ticker, "Return Autocorr (15d)"]
    raw_rets = (
        engine.df_ohlcv_raw.xs(ticker, level="Ticker")["Adj Close"]
        .pct_change()
        .loc[:date]
        .tail(16)
    )
    manual_val = raw_rets.corr(raw_rets.shift(1))
    print(
        f"Autocorr (15d): Engine={engine_val:.6f} | Manual={manual_val:.6f} | Delta={abs(engine_val-manual_val):.2e}"
    )

    # --- Range ---
    engine_val = alpha_matrix.loc[ticker, "Range Position (20d)"]
    t_df = engine.df_ohlcv_raw.xs(ticker, level="Ticker").loc[:date].tail(20)
    manual_val = (t_df["Adj Close"].iloc[-1] - t_df["Adj Low"].min()) / (
        t_df["Adj High"].max() - t_df["Adj Low"].min()
    )
    print(
        f"Range Pos (20d): Engine={engine_val:.6f} | Manual={manual_val:.6f} | Delta={abs(engine_val-manual_val):.2e}"
    )

    print("🎯 Audit Complete.")


verify_alpha_integrity(engine, "NVDA", pd.Timestamp("2026-03-31"))

In [ ]:
@dataclass
class FourDRegime:
    regime: str
    signal: str
    confidence: float
    risk_flags: list
    raw_values: dict
    metadata: dict


def analyze_4d_regime(engine, ticker, date, mode="hybrid"):
    """Logic for classifying market state based on 4 pillars."""
    # ... Implementation details ... (Truncated for readability, keep original logic in file)
    pass  # Replace with full original function during reorganization execution


# result = analyze_4d_regime(engine, "NVDA", pd.Timestamp("2026-03-31"))

## V. Builders & Heavy Lifting

In [ ]:
# --- THE MARATHON BAKE ---
CHECKPOINT_DIR = OUTPUT_DIR / "alpha_cache_v1_checkpoints"
FINAL_FILE = OUTPUT_DIR / "AlphaCache_Master_2000-2026.parquet"

# ParallelFeatureBuilder.run_marathon(
#     master_engine=engine,
#     lookbacks=[10, 21, 63, 189],
#     start_date="2000-01-01",
#     checkpoint_dir=CHECKPOINT_DIR,
#     batch_size=50,
#     num_workers=max(1, multiprocessing.cpu_count() - 2),
# )

# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

## VI. Forensic Sandbox & Utilities

In [ ]:
def cleanup_ui():
    clear_output(wait=True)
    pio.renderers.default = None
    gc.collect()
    print("🧹 Memory Cleared.")


def active_engine_audit():
    gc.collect()
    engines = [obj for obj in gc.get_objects() if type(obj).__name__ == "AlphaEngine"]
    print(f"Active AlphaEngine Instances: {len(engines)}")


active_engine_audit()

In [ ]:
# print(features_df.info())
# display(features_df.head())
# display(df_indices.tail())

In [ ]:
################################
################################
################################
################################

In [33]:
result = SU.visualize_analyzer_structure(analyzer=analyzer1)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 3))
[  4]   📂 tickers (len=3)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]   📈 initial_weights (shape=(3,))
[  9]   📂 perf_metrics (len=24)
[ 10]     🔢 full_p_gain (float)
[ 11]     🔢 full_p_sharpe (float)
[ 12]     🔢 full_p_sharpe_atrp (float)
[ 13]     🔢 full_p_sharpe_trp (float)
[ 14]     🔢 lookback_p_gain (float)
[ 15]     🔢 lookback_p_sharpe (float)
[ 16]     🔢 lookback_p_sharpe_atrp (float)
[ 17]     🔢 lookback_p_sharpe_trp (float)
[ 18]     🔢 holding_p_gain (float)
[ 19]     🔢 holding_p_sharpe (float)
[ 20]     🔢 holding_p_sharpe_atrp (float)
[ 21]     🔢 holding_p_sharpe_trp (float)
[ 22]     🔢 full_b_gain (float)
[ 23]     🔢 full_b_sharpe (float)
[ 24]     🔢 full_b_sharpe_atrp (float)
[ 25]     🔢 full_b_sharpe_trp (float)
[ 26]     🔢 lookback_b_gain (flo

In [34]:
df_snapshot = SU.peek(50, result)
df_snapshot.to_csv(OUTPUT_DIR / "df_snapshot.csv")

df_full_universe_ranking = SU.peek(51, result)
df_full_universe_ranking.to_csv(OUTPUT_DIR / "df_full_universe_ranking.csv")

 📍 INDEX: [50]
 🏷️  NAME:  universe_snapshot
 📂 PATH:  audit_pack -> debug_data -> audit_liquidity -> universe_snapshot



,ATR,ATRP,TRP,RSI,Mom_21,Consistency,IR_63,Beta_63,DD_21,AutoCorr_15,Ret_1d,Range_Pos_20,Slope_P_5,Slope_V_5,Convexity,RollingStalePct,RollMedDollarVol,RollingSameVolCount,Passed_Final
Ticker,,,,,,,,,,,,,,,,,,,
A,3.1930,0.0270,0.0293,54.3418,0.0512,0.4,-0.2499,0.8866,-0.0178,-0.1328,-0.0127,0.7072,0.0074,0.0957,-0.0003,0.0,2.2616e+08,0.0,True
AA,3.3635,0.0478,0.0378,58.2595,0.0763,0.4,0.0488,1.0655,-0.0396,0.2481,0.0004,0.7437,-0.0114,-0.0405,-0.0117,0.0,2.2739e+08,0.0,True
AAL,0.5632,0.0459,0.0432,61.6207,0.1298,0.6,-0.1121,2.0561,0.0000,-0.0966,0.0082,0.8826,0.0242,0.7186,0.0132,0.0,7.3124e+08,0.0,True
AAON,4.6710,0.0511,0.0317,56.5042,0.1521,0.6,0.0292,1.6886,-0.0248,0.0824,-0.0121,0.7558,-0.0047,-0.2545,-0.0219,0.0,7.7810e+07,0.0,False
AAPL,5.8970,0.0224,0.0224,57.4298,0.0361,0.2,0.0005,0.9787,-0.0114,-0.3195,-0.0114,0.8263,0.0050,-0.3292,0.0055,0.0,1.0939e+10,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM,3.5157,0.0406,0.0510,59.8709,0.1407,0.4,0.0293,1.0152,-0.0264,-0.5145,-0.0264,0.7558,0.0252,0.1565,0.0287,0.0,2.2758e+08,0.0,True
ZS,8.1369,0.0606,0.0520,44.3727,-0.1389,0.8,-0.1764,0.8997,-0.1389,0.1586,0.0253,0.4168,0.0325,1.1566,0.0555,0.0,4.4275e+08,0.0,True
ZTO,0.6392,0.0253,0.0221,62.2596,0.0832,0.8,0.0966,0.1060,0.0000,0.0624,0.0088,0.8685,0.0042,0.4669,-0.0014,0.0,3.4584e+07,0.0,False


 📍 INDEX: [51]
 🏷️  NAME:  full_universe_ranking
 📂 PATH:  audit_pack -> debug_data -> full_universe_ranking



,Strategy_Score,Raw_Price_Start,Raw_Price_End,Raw_TRP_Mean,Raw_ATRP_Mean,Raw_Mom_21,Raw_IR_63,Raw_Consistency,Raw_RSI,Raw_DD_21,Raw_Range_Pos_20,Raw_Slope_P_5,Raw_Slope_V_5,Raw_Convexity,Raw_AutoCorr_15,Was_Dropped
Ticker,,,,,,,,,,,,,,,,
A,-0.2257,114.540,118.250,0.0275,0.0273,0.0512,-0.2499,0.4,54.3418,-0.0178,0.7072,0.0074,0.0957,-0.0003,-0.1328,False
AA,0.7302,72.060,70.410,0.0393,0.0512,0.0763,0.0488,0.4,58.2595,-0.0396,0.7437,-0.0114,-0.0405,-0.0117,0.2481,False
AAL,-0.0730,11.130,12.270,0.0505,0.0491,0.1298,-0.1121,0.6,61.6207,0.0000,0.8826,0.0242,0.7186,0.0132,-0.0966,False
AAPL,-0.9221,255.630,263.400,0.0241,0.0227,0.0361,0.0005,0.2,57.4298,-0.0114,0.8263,0.0050,-0.3292,0.0055,-0.3195,False
ABBV,-0.3372,213.211,208.990,0.0257,0.0272,-0.0411,-0.0554,0.4,46.4258,-0.0311,0.4824,0.0044,-0.0562,0.0085,0.1622,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZBH,0.6099,91.030,94.780,0.0224,0.0235,0.0272,0.0497,0.8,58.7640,-0.0180,0.7519,0.0023,0.3469,-0.0075,-0.1092,False
ZBRA,1.5739,207.280,233.040,0.0351,0.0390,0.1147,-0.0728,0.8,64.6796,0.0000,0.9795,0.0111,1.1141,0.0048,-0.2102,False
ZM,-1.2710,80.700,86.680,0.0469,0.0364,0.1407,0.0293,0.4,59.8709,-0.0264,0.7558,0.0252,0.1565,0.0287,-0.5145,False


In [35]:
SU.export_audit_to_excel(
    audit_pack=analyzer1.last_run, filename="Audit_Verification_Report.xlsx"
)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [ ]:
SU.peek(0, result)

In [ ]:
features_df.loc["TSLA"]